In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<b><font size="5" color="red" >ch14_웹데이터수집02_동적</font></b>
## 3절. 웹사이트 동적
## 3.1 문법
```
pip install -U selenium

```
Docs : https://www.selenium.dev/documentation/

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()
driver.get("http://www.python.org")
elem = driver.find_element(By.NAME, 'q')
elem.clear()
elem.send_keys('pycon')
from selenium.webdriver.common.keys import Keys
elem.send_keys(Keys.RETURN) # enter 효과

In [ ]:
result_list = driver.find_elements(By.CSS_SELECTOR, 'h3 > a')
for result in result_list:
    print('{} - {}'.format(result.text, result.get_attribute('href')))

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(driver.page_source)
result_list = soup.select('h3 > a')
for result in result_list:
    print('{} - {}'.format(result.text, result.attrs['href']))

In [ ]:
# 셀레니움을 통해 접근한 현재 url
from urllib.parse import urlparse
current_url = driver.current_url
print('현재 주소 :', current_url)
parse_url = urlparse(current_url)
print('parse_url :',parse_url)
domain = f'{parse_url.scheme}://{parse_url.netloc}'
print(domain)

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(driver.page_source, "html.parser")
result_list = soup.select('li > h3 > a')
for result in result_list:
    print("{} - {}".format(result.text, domain+result.attrs['href']))

In [ ]:
driver.close() # 브라우저 종료

## 3.2 예제
### 1) 다음뉴스검색

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
driver = webdriver.Chrome()
driver.get('https://www.daum.net/')
# input 태그를 click
driver.find_element(By.NAME, 'q').click()
time.sleep(1)
query = input('검색어를 입력하세요')
driver.find_element(By.CSS_SELECTOR, 'input[type="text"]').send_keys(query)
driver.find_element(By.CLASS_NAME, 'btn_ksearch').click()
time.sleep(2)
# 뉴스탭 클릭 : ul.list_tab > li
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[3].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()
time.sleep(2)
news_list = []
elems = driver.find_elements(By.CSS_SELECTOR, 'div.item-title > strong.tit-g > a')
for elem in elems:
    title = elem.text
    link  = elem.get_attribute('href')
    news_list.append([title, link])
    print(title, link)

In [ ]:
# 2page로
page_div = driver.find_element(By.CSS_SELECTOR, 'div.inner_paging')
# print(page_div.text)
next_page = page_div.find_element(By.LINK_TEXT, '2')
next_page.click()

#### 페이지 처리
- 다음 뉴스 검색 : query, page수

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
driver = webdriver.Chrome()
driver.get('https://www.daum.net/')
# input 태그를 click
time.sleep(1)
driver.find_element(By.NAME, 'q').click()
query = input('검색어를 입력하세요')
driver.find_element(By.CSS_SELECTOR, 'input[type="text"]').send_keys(query)
driver.find_element(By.CLASS_NAME, 'btn_ksearch').click()
time.sleep(2)
# 뉴스탭 클릭 : ul.list_tab > li
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[3].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()
time.sleep(2)
news_list = []
pages = 3
for page in range(1, pages+1):
    elems = driver.find_elements(By.CSS_SELECTOR, 'div.item-title > strong.tit-g > a')
    for elem in elems:
        title = elem.text
        link  = elem.get_attribute('href')
        news_list.append([title, link])
        # print(title, link)
    # 2page로
    page_div = driver.find_element(By.CSS_SELECTOR, 'div.inner_paging')
    # print(page_div.text)
    next_page = page_div.find_element(By.LINK_TEXT, str(page+1) )
    next_page.click()
    time.sleep(1.5)
    print(f'~~~ 현재 {page}페이지 데이터 수집 중입니다 ~ ~')
driver.close()
import pandas as pd
df = pd.DataFrame(news_list, columns=['title','link'])
df.to_csv('data/ch14_daum.csv', index=False)


### 2) 맞춤법 검사기
- 네이버 맞춤법 검사기 이용

In [ ]:
# ch14_맞춤법전.txt를 300자 이내로 자르기
with open('data/ch14_맞춤법전.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print('총글자 수 :', len(text))
ready_list = [] # 맞춤법 검사할 text 내용(300자 이내로 list)
while(len(text) > 300):
    temp = text[:300]
    new_line_char_index = temp.rfind('\n')
    print(new_line_char_index)
    ready_list.append(text[:new_line_char_index])
    text = text[new_line_char_index:]
ready_list.append(text)
[ready[:10] for ready in ready_list]

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
driver = webdriver.Chrome()
time.sleep(0.5)
driver.get('https://www.naver.com/')
input_elem = driver.find_element(By.CSS_SELECTOR, 'input[name="query"]')
input_elem.send_keys('맞춤법 검사기')
input_elem.send_keys(Keys.RETURN)
time.sleep(0.5)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
textarea.send_keys(ready_list[0])

button = driver.find_element(By.CLASS_NAME, 'btn_check')
button.click()
time.sleep(2)

soup = BeautifulSoup(driver.page_source, "html.parser")
result = soup.select_one('p._result_text.stand_txt').text
result

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
driver = webdriver.Chrome()
time.sleep(0.5)
driver.get('https://www.naver.com/')
input_elem = driver.find_element(By.CSS_SELECTOR, 'input[name="query"]')
input_elem.send_keys('맞춤법 검사기')
input_elem.send_keys(Keys.RETURN)
time.sleep(0.5)
results = '' # 맞춤법 검사 후 내용
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')

for ready in ready_list:

    textarea.send_keys(Keys.CONTROL, 'a') # control + a
    textarea.send_keys(ready)

    button = driver.find_element(By.CLASS_NAME, 'btn_check')
    button.click()
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    result = soup.select_one('p._result_text.stand_txt').text
    results += result + '\n'
driver.close()
with open('data/ch14_맞춤법후.txt', 'w', encoding='utf-8') as f:
    f.write(results)
    
    
    #원본

In [ ]:
# ch14_맞춤법후.txt를 1000자이내로 자르기
with open('data/ch14_맞춤법후.txt', 'r', encoding='utf-8') as f:
    text = f.read()
ready_list = [] # 맞춤법 검사할 text 내용(300이자 이내로 list)
while(len(text) > 1000):
    temp = text[:1000]
    new_line_char_index = temp.rfind('\n')
    ready_list.append(text[:new_line_char_index])
    text = text[new_line_char_index:]
ready_list.append(text)
print('크롤링 할 text 수 :',[len(ready) for ready in ready_list])
# 크롤링
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
driver = webdriver.Chrome()
time.sleep(0.5)
driver.get('https://translate.kakao.com/')
textarea = driver.find_element(By.CSS_SELECTOR, 'textarea.translate_area.area_item.translate_many')
button = driver.find_element(By.CSS_SELECTOR, 'div.btn_item')
results = ''
for idx, ready in enumerate(ready_list):
    print( f'{round(idx/len(ready_list) * 100, 2)}% 번역 중입니다')
    textarea.clear()
    textarea.send_keys(ready)
    button.click()
    time.sleep(1)

    # soup = BeautifulSoup(driver.page_source, "html.parser")
    # result = soup.select_one('div.result_area.translate_many').text
    result = driver.find_element(By.CSS_SELECTOR, 'div.result_area.translate_many').text
    results += result + '\n'
    
# 번역한 결과 파일 출력
with open('data/ch14_자동화영어번역본.txt', 'w', encoding='utf-8') as f:
    f.write(results)
print('번역 완료')

In [ ]:
### 작업중 ###
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

with open('data/ch14_맞춤법후.txt', 'r', encoding='utf-8') as f:
    text = f.read()
ready_list = [] # 맞춤법 검사할 text 내용(300이자 이내로 list)

while(len(text) > 300):
    temp = text[:300]
    new_line_char_index = temp.rfind('\n')
    print(new_line_char_index)
    ready_list.append(text[:new_line_char_index])
    text = text[new_line_char_index:]
ready_list.append(text)
results = ""

for ready in ready_list:
    input_area = driver.find_element(By.CSS_SELECTOR, 'input[name="q"]')
    input_area.clear()
    input_area.send_keys(ready)
    
    button = driver.find_element(By.CLASS_NAME, 'inner')
    button.click()
    time.sleep(2)

    output_area = driver.find_element(By.CSS_SELECTOR, "result_area.area_item.txt_eng.translate_many")
    translated = output_area.text
    results += translated + '\n'

